# Session IX: Advanced Object-Oriented Programming Techniques

Apply inheritance, class methods, static methods, and polymorphism to make flexible, powerful class structures.

In this session you will practice:
- extending a base class with subclasses
- using `super()`
- adding `@classmethod` utilities
- organizing validation with `@staticmethod`
- writing code that works with multiple related object types


## Exercise 1: Transaction Inheritance
**Objective:** Extend a base class using inheritance.

Create:
- a base `Transaction` class
- a `RecurringTransaction` subclass

**Tasks:**
1. `Transaction` should store:
   - `amount`
   - `category`
   - `description`
2. Add a `describe()` method in `Transaction`.
3. `RecurringTransaction` should inherit from `Transaction` and add:
   - `frequency` (for example: `"weekly"` or `"monthly"`)
4. Use `super()` inside the subclass constructor.
5. Override `describe()` so recurring transactions show their frequency too.
6. Add an `annual_cost()` method to `RecurringTransaction`.


In [ ]:
class Transaction:
    """Represent a one-time transaction."""
    def __init__(self, amount, category, description):
        self.amount = amount
        self.category = category
        self.description = description

    def describe(self):
        return (f"This transaction has:\n"
                f"  Amount: {self.amount}\n"
                f"  Category: {self.category}\n"
                f"  Description: {self.description}")


class RecurringTransaction(Transaction):
    """Represent a recurring transaction such as rent or subscriptions."""
    PERIODS_PER_YEAR = {"weekly": 52, "monthly": 12, "yearly": 1}

    def __init__(self, amount, category, description, frequency):
        super().__init__(amount, category, description)
        self.frequency = frequency

    def describe(self):
        return super().describe() + f"\n  Frequency: {self.frequency}"

    def annual_cost(self):
        """Estimate the annual cost based on the frequency."""
        # weekly = 52, monthly = 12, yearly = 1
        return self.amount * self.PERIODS_PER_YEAR[self.frequency]


# --- TEST ---
t1 = Transaction(18.50, "Food", "Lunch")
t2 = RecurringTransaction(950, "Housing", "Rent", "monthly")
t3 = RecurringTransaction(12, "Streaming", "Music subscription", "monthly")

print(t1.describe())
print(t2.describe())
print(t3.annual_cost())

---
## Exercise 2: Expense Summary with Class Methods
**Objective:** Use class methods for logic that belongs to the class as a whole.

This is a new, simpler `Transaction` class (only `amount` and `category`), independent of Exercise 1.

**Tasks:**
1. Create a `Transaction` class with:
   - `amount`
   - `category`
2. Add a `@classmethod` called `from_dict(data)` that builds a transaction from a dictionary by returning `cls(...)` (an alternative constructor).
3. Add a `@classmethod` called `summarize_by_category(transactions)` that returns a dictionary of totals by category.
4. Add a `@classmethod` called `total_amount(transactions)` that returns the total amount across all transactions.


In [ ]:
class Transaction:
    """Represent a transaction that can also be built or summarized at the class level."""
    def __init__(self, amount, category):
        self.amount = amount
        self.category = category

    @classmethod
    def from_dict(cls, data):
        """Create a Transaction from a dictionary like {'amount': 12.5, 'category': 'Food'}."""
        return cls(data["amount"], data["category"])

    @classmethod
    def summarize_by_category(cls, transactions):
        """Return a dict like {'Food': 30, 'Transport': 10}."""
        totals = {}
        for t in transactions:
            totals[t.category] = totals.get(t.category, 0) + t.amount
        return totals

    @classmethod
    def total_amount(cls, transactions):
        """Return the total amount across all transactions."""
        return sum(t.amount for t in transactions)


# --- TEST ---
raw = [
    {"amount": 12.5, "category": "Food"},
    {"amount": 40, "category": "Transport"},
    {"amount": 25, "category": "Food"},
]

transactions = [Transaction.from_dict(item) for item in raw]
print(Transaction.summarize_by_category(transactions))
print(Transaction.total_amount(transactions))

---
## Exercise 3: Validation Utilities with Static Methods
**Objective:** Use static methods to keep validation logic related to a class.

Create a `Contact` class with utility validation methods.

**Tasks:**
1. Add a `@staticmethod` called `is_valid_email(email)`.
2. Add a `@staticmethod` called `is_valid_phone(phone)`.
3. Add a `@staticmethod` called `normalize_name(name)` that strips spaces and applies title case.
4. In `__init__()`, normalize the name before storing it.


In [ ]:
class Contact:
    """Represent one contact with built-in validation helpers."""
    def __init__(self, name, phone, email):
        self.name = Contact.normalize_name(name)
        self.phone = phone
        self.email = email

    @staticmethod
    def is_valid_email(email):
        # basic validation: must contain '@' and '.'
        return "@" in email and "." in email

    @staticmethod
    def is_valid_phone(phone):
        # accept digits, spaces, dashes, and parentheses
        allowed = "0123456789 -()"
        has_digit = any(ch.isdigit() for ch in phone)
        return has_digit and all(ch in allowed for ch in phone)

    @staticmethod
    def normalize_name(name):
        # split() also collapses repeated inner spaces
        return " ".join(name.split()).title()


# --- TEST ---
print(Contact.is_valid_email("alice@email.com"))
print(Contact.is_valid_email("wrong-email"))
print(Contact.is_valid_phone("(555) 123-4567"))
print(Contact.is_valid_phone("abc123"))

c = Contact("   aLiCe   sMiTh  ", "555-1234", "alice@email.com")
print(c.name)

---
## Exercise 4: Contact Book 2.0
**Objective:** Build a more complete contact book using OOP plus validation.

Create:
- a `Contact` class
- a `ContactBook` class

**Tasks:**
1. `Contact` should store `name`, `phone`, and `email`.
2. Add a `@staticmethod` called `is_valid_email(email)`.
3. In `ContactBook`, implement:
   - `add_contact(contact)`
   - `add_contact_data(name, phone, email)` → validates before creating the object
   - `search(name)`
   - `all_contacts()`
4. Store contacts in a dictionary by lowercase name.
5. Return `True` or `False` from `add_contact_data()` depending on whether the data is valid.


In [ ]:
class Contact:
    def __init__(self, name, phone, email):
        self.name = name
        self.phone = phone
        self.email = email

    @staticmethod
    def is_valid_email(email):
        return "@" in email and "." in email

    def __str__(self):
        return f"{self.name} | {self.phone} | {self.email}"


class ContactBook:
    def __init__(self):
        self.contacts = {}

    def add_contact(self, contact):
        self.contacts[contact.name.lower()] = contact

    def add_contact_data(self, name, phone, email):
        if not Contact.is_valid_email(email):
            return False
        self.add_contact(Contact(name, phone, email))
        return True

    def search(self, name):
        return self.contacts.get(name.lower())

    def all_contacts(self):
        return list(self.contacts.values())


# --- TEST ---
book = ContactBook()
print(book.add_contact_data("Alice", "555-1234", "alice@email.com"))
print(book.add_contact_data("Bob", "555-5678", "wrong-email"))

print(book.search("alice"))
for contact in book.all_contacts():
    print(contact)

---
## Exercise 5: Polymorphic Budget Planner
**Objective:** Use polymorphism so one loop can handle different transaction types.

Create:
- a base `Transaction` class
- a `OneTimeTransaction` subclass
- a `RecurringTransaction` subclass
- a `BudgetManager` class

**Tasks:**
1. Every transaction class should implement `monthly_cost()`. In the base `Transaction` class (which stores `amount` and `description`), it is a placeholder that must `raise NotImplementedError`, so a subclass that forgets to override it fails loudly.
2. `OneTimeTransaction.monthly_cost()` should return its amount.
3. `RecurringTransaction.monthly_cost()` should convert:
   - weekly → amount × 52 / 12
   - monthly → amount
   - yearly → amount / 12
4. `BudgetManager` should store mixed transaction objects.
5. Add methods:
   - `add_transaction(transaction)`
   - `total_monthly_cost()`
   - `summary_lines()` → return a list of strings using polymorphism


In [ ]:
class Transaction:
    """Base class for budget transactions."""
    def __init__(self, amount, description):
        self.amount = amount
        self.description = description

    def monthly_cost(self):
        raise NotImplementedError("Subclasses must implement monthly_cost()")

    def summary_line(self):
        return f"{self.description}: {self.monthly_cost():.2f}/month"


class OneTimeTransaction(Transaction):
    def monthly_cost(self):
        return self.amount


class RecurringTransaction(Transaction):
    def __init__(self, amount, description, frequency):
        super().__init__(amount, description)
        self.frequency = frequency

    def monthly_cost(self):
        if self.frequency == "weekly":
            return self.amount * 52 / 12
        if self.frequency == "monthly":
            return self.amount
        if self.frequency == "yearly":
            return self.amount / 12
        raise ValueError(f"Unknown frequency: {self.frequency}")

    def summary_line(self):
        return f"{super().summary_line()} ({self.frequency})"


class BudgetManager:
    def __init__(self):
        self.transactions = []

    def add_transaction(self, transaction):
        self.transactions.append(transaction)

    def total_monthly_cost(self):
        return sum(t.monthly_cost() for t in self.transactions)

    def summary_lines(self):
        return [t.summary_line() for t in self.transactions]


# --- TEST ---
manager = BudgetManager()
manager.add_transaction(OneTimeTransaction(120, "Books"))
manager.add_transaction(RecurringTransaction(950, "Rent", "monthly"))
manager.add_transaction(RecurringTransaction(30, "Gym", "monthly"))
manager.add_transaction(RecurringTransaction(15, "Cloud Storage", "yearly"))

for line in manager.summary_lines():
    print(line)

print("Estimated monthly total:", round(manager.total_monthly_cost(), 2))